In [ ]:
import json
import uuid
from qdrant_client import QdrantClient
from qdrant_client.models import VectorParams, Distance, PointStruct
from sentence_transformers import SentenceTransformer

from data import doctor_info, Hospital_info


In [ ]:
client = QdrantClient("http://localhost", port=6334)
model = SentenceTransformer("all-MiniLM-L6-v2") 
collection_name = "Crail_data"

In [ ]:
# client.recreate_collection(
#     collection_name=collection_name,
#     vectors_config=VectorParams(size=384, distance=Distance.COSINE)
# )

In [ ]:
def flatten_doc(data):
    flat_text = []
    for key, value in data.items():
        formatted_key = key.replace('_', ' ').capitalize()
        
        if isinstance(value, dict):
            nested = flatten_doc(value)
            flat_text.append(f"{formatted_key}: {nested}")
        elif isinstance(value, list):
            joined = ", ".join(str(v) for v in value)
            flat_text.append(f"{formatted_key}: {joined}")
        else:
            flat_text.append(f"{formatted_key}: {value}")
    return " | ".join(flat_text)


In [ ]:
def generate_and_save_embeddings(user_info, data_list):
    points = []
    for data in data_list:
        doc_id = str(uuid.uuid4())
        flattened = flatten_doc(data)

        embedding = model.encode(flattened).tolist()

        point = PointStruct(
            id=doc_id,
            vector=embedding,
            payload={
                "user_id" : user_info["user_id"],
                "data": flattened,
            }
        )
        points.append(point)
    
    client.upsert(collection_name=collection_name, points=points)



In [ ]:
user_info = {
    "user_id": 456,
}

with open("E:\\Crail 2025\\logistics_shipments.json", "r", encoding="utf-8") as file:
    json_doc = json.load(file)


generate_and_save_embeddings(user_info, json_doc)